# TNF-α prescreen — walkthrough

Screen a batch of designed sequences for prior art and known anti-TNF binders, locally.

Everything here runs on your own machine. The curated binder set ships inside the package,
so §1–§4 need **no downloads at all**; the public-corpus search in §5 needs the reference
databases built first (`bash scripts/build_reference_dbs.sh ~/prescreen-dbs`).

Setup: `uv pip install -e .` plus an `mmseqs` binary on `$PATH`.

In [ ]:
import json, os, subprocess, tempfile
from pathlib import Path

import pandas as pd
from prescreen import Config, refdb, screen, to_rows

print("mmseqs:", subprocess.run(["mmseqs", "version"], capture_output=True, text=True).stdout.strip())

## 1. A batch to screen

The screen is **batched** — one MMseqs2 call per database for the whole input — so database
load dominates and ten sequences cost about what one does. Submit everything at once rather
than looping.

Batching is also what makes the `batch_duplicate` flag meaningful: it reports submissions
that duplicate **each other** within the same run.

The batch below is deliberately mixed, so each arm of the filter shows its hand.

In [ ]:
# Adalimumab (Humira) heavy chain — a confirmed anti-TNF binder, id TNFB00005
ADALIMUMAB_VH = (
    "EVQLVESGGGLVQPGRSLRLSCAASGFTFDDYAMHWVRQAPGKGLEWVSAITWNSGHIDYADSVEGRFTISRDNAKNSLY"
    "LDMNSLRAEDTAVYYCAKVSYLSTASSLDYWGQGTLVTVSSASTKGPSVFPLAPSSKSTSGGTAALGCLVKDYFPEPVTV"
    "SWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTQTYICNVNHKPSNTKVDKKI"
)

batch = {
    "known_binder":       ADALIMUMAB_VH,
    "duplicate":          ADALIMUMAB_VH,                                 # submitted twice
    "framework_edited":   ADALIMUMAB_VH[:40] + "GSGSGSGSGS" + ADALIMUMAB_VH[50:],
    "fragment":           ADALIMUMAB_VH[95:112],                         # 17-mer
    "public_miniprotein": "GSMDKEELVQKAKLAEQAERYDDMAACMKSVTEQGAELSNEERNLLSVAYKNVVGARRSSWRV",
}
{k: len(v) for k, v in batch.items()}

## 2. The known-binder arm, with no databases

`skip_prior_art=True` runs the TNF-specific half only: whole sequence against the curated
binder set, the paratope comparison, and the verbatim-fragment check. Nothing to download —
the 4,964-entry reference set is in `prescreen/data/`.

In [ ]:
out  = screen(batch, cfg=Config.from_env(), skip_prior_art=True)
rows = to_rows(out)

pd.DataFrame(rows)[[
    "id", "length", "category", "verdict", "flags",
    "target_region_identity", "target_named_agent", "target_binder_status", "cluster_size",
]].set_index("id")

What that table says, and why each row is there:

| id | category | verdict | why |
|---|---|---|---|
| `known_binder` | `vh_domain` | `known_target_binder` | whole sequence **and** paratope match a confirmed binder; `target_named_agent` names it |
| `duplicate` | `vh_domain` | `known_target_binder` | same molecule submitted twice |
| `framework_edited` | `vh_domain` | `known_target_binder` | **the point of the filter** — 10 framework residues replaced, CDR3 untouched, so it is still the same binder |
| `fragment` | `peptide` | `known_target_binder` | a 17-mer, caught by `verbatim_target_fragment`. Default MMseqs2 settings return *nothing* for a query this short; the sensitive short-query pass is what finds it |
| `public_miniprotein` | `miniprotein` | `pass` | nothing to do with TNF, so the known-binder arm is silent. The prior-art arm in §5 is what has something to say about it |

All four adalimumab-derived entries also carry `batch_duplicate` with `cluster_size` 4 —
clustering is on sequence, so the fragment clusters with the chain it was cut from. Within
a real submission batch that is the signal you want: one design submitted five ways.

Try the converse: replace the CDR3 (around residues 95–112) and leave the framework alone,
and the entry passes — reusing a published scaffold with a genuinely new paratope is
allowed, and is the whole reason the paratope arm exists.

In [ ]:
# How the category was decided, and which region was compared
rec = out["results"]["known_binder"]
print("category :", rec["category"], "/", rec["region_kind"], f"(confidence {rec['category_confidence']})")
print("focus    :", rec["focus_chain"], rec["focus_region"])
print("evidence :", json.dumps(rec["category_evidence"])[:300])

## 3. The evidence behind a call

Every threshold and every intermediate is in the record, for when someone disputes a
verdict.

In [ ]:
print(json.dumps(out["results"]["known_binder"]["reasons"], indent=2)[:1800])

## 4. From the shell

The CLI takes FASTA or CSV and writes `report.csv` (flat, one row per submission) plus
`report.json` (everything).

```bash
prescreen examples/submissions.fasta --skip-prior-art -o report/
prescreen submissions.csv -o report/ --flagged-only
```

## 5. Adding the prior-art arm

This is the half that answers "is this sequence already public". It needs the reference
databases:

```bash
bash scripts/build_reference_dbs.sh ~/prescreen-dbs      # everything but uspto
export PRESCREEN_DB_ROOT=~/prescreen-dbs
```

The four small arms (THPdb, Thera-SAbDab, PLAbDab-nano, Proteinbase) build in about a
minute and need no credentials. PDB, SwissProt and PLAbDab are ~1.7 GB together; the USPTO
patent arm is a 9.3 GB download, and it is searched on every run.

In [ ]:
# Which arms are actually on disk? Run this before trusting a "no prior art" verdict.
pd.DataFrame(
    [{"arm": k, "built": bool(v), "path": v or ""} for k, v in refdb.available().items()]
).set_index("arm")

In [ ]:
cfg = Config.from_env()          # picks up $PRESCREEN_DB_ROOT
# cfg.include_patent_arm = False   # uncomment to drop the USPTO arm (it is on by default)

full = screen(batch, cfg=cfg)
pd.DataFrame(to_rows(full))[[
    "id", "verdict", "flags", "prior_art_db", "prior_art_hit", "prior_art_similarity",
]].set_index("id")

### The one that bites

An arm whose database is **not** on disk is skipped *silently* — and a skipped arm reads
exactly like a sequence with no prior art, which is the answer a prescreen must never give
by accident. Nothing warns you on stdout. Check it explicitly:

In [ ]:
pa = full["results"]["known_binder"]["prior_art"]
print("searched:", pa["arms_searched"])
print("missing :", pa["arms_missing"] or "none")
if pa["arms_missing"]:
    print("\n⚠ a 'no prior art' verdict in this run is weaker than it looks")

## 6. A different target

Nothing above is TNF-specific except the contents of `prescreen/data/`. Point `--target` (or
`target_fasta=`) at your own curated binder set — one FASTA, optionally a CSV of per-entry
provenance — and the same two arms run against it.

```python
screen(batch, cfg=cfg, target_fasta="my_target_binders.fasta",
       target_metadata="my_target_binders.csv")
```

Cut points live in `Config` and every one is overridable from the environment
(`PRESCREEN_REGION_MATCH=0.85`, …). What each threshold means and the calibration behind it
are in `PRESCREEN_SPEC.md`.